# 🔧 Otto Crane — The Gruff Engineer
## Voice Aging MVP

**Character Arc:** Otto Crane — brash working-class genius inventor who becomes a cantankerous legend.

**Personality Core:** a loud, opinionated, brilliantly practical male engineer — no patience for nonsense, strong regional working-class accent implied in rhythm, straight-talking to the point of rudeness, heart of gold buried under layers of complaints

In [ ]:
!pip install -q qwen-tts soundfile accelerate transformers

In [ ]:
import os
import gc
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
personality_core = "a loud, opinionated, brilliantly practical male engineer — no patience for nonsense, strong regional working-class accent implied in rhythm, straight-talking to the point of rudeness, heart of gold buried under layers of complaints"

character_data = [
    {
        "stage": "youth",
        "age": 22,
        "title": "The Upstart",
        "modifier": "loud and brash, working-class chip on the shoulder, impatient genius who hasn't learned to explain himself yet, fast and aggressive speech",
        "lines": [
            "I don't need a degree to know this design is wrong. I need five minutes and a wrench. Move.",
            "The people with the fancy qualifications built this. I'm the one fixing it. I think that says everything that needs saying.",
            "I'm not being rude. I'm being efficient. If you want polite, hire someone slower."
        ]
    },
    {
        "stage": "prime",
        "age": 40,
        "title": "The Master",
        "modifier": "still direct and opinionated, but the genius now fully visible, gravelly authority, professional respect earned and occasionally weaponized",
        "lines": [
            "Twenty years in the field and I've never seen a problem that couldn't be solved by applying more intelligence to the right place.",
            "The difference between an amateur and a professional is that the professional knows which rules to break and when.",
            "I don't take on problems I can't solve. I haven't turned down a problem yet. Draw your own conclusions."
        ]
    },
    {
        "stage": "middle",
        "age": 57,
        "title": "The Veteran",
        "modifier": "slower and gruffer, complaints multiplied but genuine warmth for his craft visible, the impatience now partly performative, deeply principled beneath the bluster",
        "lines": [
            "Thirty-five years of building things, and the younger generation still tries to make things more complicated than they need to be. Some things don't change.",
            "I've mentored forty engineers. Thirty-seven of them are better than anyone gives them credit for. Three of them are geniuses. I'm proud of all of them. Don't tell them.",
            "I've been offered more money than I care to think about to work for people I didn't respect. Said no every time. That is the entirety of my financial philosophy."
        ]
    },
    {
        "stage": "elder",
        "age": 72,
        "title": "The Legend",
        "modifier": "gravelly and slow, cantankerous but fundamentally warm, complaints have become a kind of poetry, legend's authority requiring no performance",
        "lines": [
            "I've forgotten more engineering than most people have ever learned. I try not to hold that against them. I mostly succeed.",
            "Fifty years ago I fixed something that everyone said was unfixable. Sixty years ago, I fixed two. People kept saying it was impossible. I kept ignoring them. Still works.",
            "I'm not a legend. I'm an old man who was very good at one thing and stubborn enough to keep doing it. Call it what you want."
        ]
    }
]

In [ ]:
print("Loading Qwen3-TTS VoiceDesign model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
model = Qwen3TTSModel.from_pretrained(model_id, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")
print("Model loaded successfully!")

In [ ]:
all_audio_data = []
sr = None
char_id = "otto"

for stage_info in character_data:
    stage = stage_info["stage"]
    title = stage_info["title"]
    instruct = f"{personality_core}, {stage_info['modifier']}"
    print(f"\n--- Stage: {title} ({stage_info['age']}) ---")
    print(f"Instruction: {instruct}\n")
    
    stage_audio = []
    for i, line in enumerate(stage_info["lines"]):
        print(f"Generating Line {i+1}...")
        wavs, cur_sr = model.generate_voice_design(line, language="English", instruct=instruct)
        wav_data = wavs[0]
        if sr is None:
            sr = cur_sr
        
        filename = f"{char_id}_{stage}_{i+1}.wav"
        sf.write(filename, wav_data, sr)
        
        print(f"Line {i+1}: \"{line}\"")
        display(Audio(filename))
        
        stage_audio.append(wav_data)
        
    all_audio_data.append(stage_audio[0])
    
    # Cleanup VRAM
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Life story montage: line 1 from each stage concatenated with 2.5s silence
print("\n--- Life Story Montage ---")
silence_duration = 2.5
if sr is not None:
    silence_samples = int(sr * silence_duration)
    silence = np.zeros(silence_samples, dtype=np.float32)
    
    montage = []
    for i, audio in enumerate(all_audio_data):
        montage.append(audio)
        if i < len(all_audio_data) - 1:
            montage.append(silence)
            
    montage_audio = np.concatenate(montage)
    montage_filename = f"{char_id}_montage.wav"
    sf.write(montage_filename, montage_audio, sr)
    print(f"Generated: {montage_filename}")
    display(Audio(montage_filename))
else:
    print("No audio was generated.")

In [ ]:
print("\n--- Emotional Range Test ---")
test_line = "You want me to fix this? After what you did to it?"

emotions = [
    ("Angry", "furious, exploding with anger, loud and shouting, losing patience completely"),
    ("Resigned", "deeply tired, exhausted sighing, resigned to the stupidity of the world, rubbing temples"),
    ("Excited", "secretly thrilled by the challenge, barely contained excitement, intense focus")
]

for emotion_name, emotion_mod in emotions:
    print(f"\nEmotion: {emotion_name}")
    instruct = f"{personality_core}, {emotion_mod}"
    print(f"Instruction: {instruct}")
    
    wavs, cur_sr = model.generate_voice_design(test_line, language="English", instruct=instruct)
    
    filename = f"{char_id}_emotion_{emotion_name.lower()}.wav"
    sf.write(filename, wavs[0], cur_sr)
    print(f"Line: \"{test_line}\"")
    display(Audio(filename))
    
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
!zip -q otto_crane_outputs.zip otto_*.wav
from google.colab import files
files.download('otto_crane_outputs.zip')
print("Downloaded outputs zip.")